第11回講義
========

音声データの可視化
--------------

音楽や人の声などの音声データは、コンピュータ上で処理をするためにサンプリング(量子化)というアナログ/デジタル変換(A/D)によって数値化(デジタル化)されます。通常のCDなどの音源のサンプリング周波数は44.1kHz(44100Hz)です。

音声データのフォーマットには圧縮音源も含めて様々なものがありますが、wavフォーマットの音源をPythonに読み込むには、`wave`モジュールを用います。

In [ ]:
import wave

wf = wave.open('piano.wav', 'r')

`wave`モジュールで読み込んだオブジェクトから、wavフォーマット内に記録されているサンプリング周波数やサンプル幅といったプロパティを表示させることができます。

In [ ]:
print("サンプリング周波数: ", wf.getframerate())
print("サンプル幅: ", wf.getsampwidth())

waveオブジェクトからデータを読み出す際には、`readframes()`メソッドを用いる。`readframes()`メソッドでは、読み出すフレーム数を指定するが、引数に`-1`を指定すると、記録されているすべてのフレーム(データ)を読み出す。

In [ ]:
data = wf.readframes(-1)

データはバイナリ形式で格納されているので、numpyの`frombuffer`関数を使って、ndarrayに変換する必要がある。
ステレオの場合は、左チャンネルと右チャンネルのデータが交互に格納されているので、1つおきに取り出せば左チャンネルのみとなる。

<mark>練習1</mark> `piano.wav`から音声データを読み出し、左チャンネルの音声データをプロットしなさい。また、プロットの一部を拡大して表示させなさい。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import wave

wf = wave.open('piano.wav', 'r')

data = wf.readframes(-1)
data = np.frombuffer(data, dtype=np.int16)

# 左チャンネル全ての波形をプロット
plt.plot(data[::2])
plt.show()

plt.plot(data[1000:2000:2])

スペクトル分析
-----------

時系列データにどのような周期(周波数成分)が入っているかを調べる方法を**スペクトル分析**といいます。スペクトル分析ではフーリエ級数という考えかたを用います。

周期$T$を持つある関数$f(x)$があったとき、$f(x)$は以下のようなフーリエ級数展開を持ちます。
$$f(x)=\frac{a_0}{2}+\sum_{n=1}^\infty \left(a_n \cos\frac{2\pi n x}{T}+b_n\sin\frac{2\pi n x}{T}\right)$$
ここで、展開の係数$a_n$、$b_n$は、
$$a_n = \frac{2}{T}\int_0^T f(x)\cos\frac{2\pi n x}{T} dx$$
$$b_n = \frac{2}{T}\int_0^T f(x)\sin\frac{2\pi n x}{T} dx$$
で与えられます。

<mark>練習2</mark> 簡単のため$T=1$として、$\sin 2\pi x$、$\sin 4\pi x$、$\sin 6\pi x$を色々な係数$b_1$, $b_2$, $b_3$で重ね合わせて表示してみなさい。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

x = np.linspace(0, 3, 100)

b1 = 5.0
b2 = 1.0
b3 = 1.5

y = b1*np.sin(2*np.pi*x) + b2*np.sin(4*np.pi*x) + b3*np.sin(6*np.pi*x)
plt.plot(x, y)

### 離散フーリエ変換

フーリエ級数展開のどの係数が大きいかを調べることによって、その関数(時系列データ)の周期性を調べることができます。

例えば、人の声や楽器の音はさまざまな周波数成分(sin、cos関数)が合成されたものですが、フーリエ級数展開の主要な部分で全体の音程(音の高さ)が規定されています。

コンピュータ上でフーリエ級数展開の無限個の係数を全て求めることはできないので、有限個離散フーリエ変換といったアルゴリズムを用います。フーリエ級数を求めるのは電子工学や通信など工業的にも非常に応用範囲が広いので、高速フーリエ変換やそれ専用のチップなど様々なアルゴリズムや手法が開発されています。

離散フーリエ変換では係数を複素数とすることで、$\sin$関数と$\cos$関数での展開を同時に行います。
$$
\hat{f}_k = \sum_{n=0}^{N-1}f_n e^{-i \omega_k n} 
$$
ここで、$\omega_k =\frac{2\pi k}{N}$ ($k=0,1,\cdots,N-1$)です。($e^{i\theta}=\cos\theta+i\sin\theta$であることを思い出す。)

係数が複素数なので、フーリエ変換したものの$k$での値を調べる際には、その絶対値($z=a+ib$のとき、$|z|=\sqrt{a^2+b^2}$)を計算します。

<mark>練習3</mark> $f_n=\sin \frac{4\pi n}{10}$ ($n=0,1,\cdots,9$)として、$\hat{f}_k$ ($k=0,1,\cdots,9$)の大きさ(絶対値)をリストとして求めなさい。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 10
n = np.arange(0, N)

fn = np.sin(4*np.pi*n/N)

fk = []

for k in range(N):
    omega_k = 2*np.pi*k/N
    
    s = 0
    
    for i in range(N):
        s += fn[i] * np.exp(-1j*omega_k*i)
    
    fk.append(s)

plt.plot(n, fn)

plt.plot(n, np.abs(fk))

一般に、データ列$f_n$に対して、$f_n=f_{n+p}$という周期$p$が存在するとき、$\hat{f}_{N/p}$と$\hat{f}_{N(1-1/p)}$に大きなピークが現れます。($N$は$p$で割り切れて、$N/p=m$と仮定する。) 離散フーリエ変換を行なって、どのような$k$の値でピークが現れるかを調べることによって、元のデータ列の周期を読み取ることができます。

証明:
$\omega_{N/p}=\frac{2\pi}{p}$のとき、
$$
\begin{split}
\hat{f}_{N/p}&=\sum_{n=0}^{N-1}f_n e^{-2\pi i n/p}\\
 &= \sum_{n=0}^{p-1}f_n e^{-2\pi i n/p} + \sum_{n=p}^{2p-1}f_n e^{-2\pi i n/p}+\cdots\\
 &= m\sum_{n=0}^{p-1}f_n e^{-2\pi i n/p}
\end{split}
$$
一方、$\omega_k\neq\frac{2\pi}{p}$のとき、
$$
\begin{split}
\hat{f}_k&=\sum_{n=0}^{N-1}f_n e^{-2\pi i \omega_k n}\\
 &= \sum_{n=0}^{p-1}f_n e^{-2\pi i n/p} + \sum_{n=p}^{2p-1}f_n e^{-2\pi i n/p}+\cdots\\
 &= (1+e^{-2\pi i\omega_k p}+e^{-4\pi i\omega_k p}+\cdots)\sum_{n=0}^{p-1}f_n e^{-2\pi i n/p}\\
\simeq 0
\end{split}
$$
また、$\omega_{N(1-1/p)}=2\pi -\frac{2\pi}{p}$なので、同様の現象が起こります。


離散フーリエ変換のための関数はnumpyモジュールなどに備わっているので、それを使うことで簡単にフーリエ級数展開を用いた解析(フーリエ解析)を行うことができます。

### 音声ファイルのフーリエ変換

<mark>練習4</mark> numpyの中の`fft.fft()`関数を用いて音声データ`piano.wav`について、離散フーリエ変換を行い、その大きさ(絶対値)をプロットしなさい。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import wave

wf = wave.open('piano.wav', 'r')

data = wf.readframes(-1)
data = np.frombuffer(data, dtype=np.int16)

fft = np.fft.fft(data[::2])

plt.plot(np.abs(fft))

このままでは周波数などの情報が読み取れないので、上のプロットの意味を考えてみましょう。まず、周期$p$に関して、$\hat{f}$では$N/p$と$N(1-1/p)$が対称になるので、その半分は不要です。

次に、もともとの音声データはCDと同じサンプリング周波数(44100Hz)で記録されているので、データ列は1/44100秒間隔で収録されているということになります。そこに$p$データごとの周期が見られたとすると、実際の時間での周期は、
$$
T = p/44100
$$
ということになります。あるいは、この周期に対応する周波数は、
$$
\nu = 44100/p
$$
です。

一方、フーリエ変換した後の$k=N/p$にピークが見られたとすれば、$p=N/k$なので、上の式に代入して、周波数が
$$
\nu = 44100 \times k / N
$$
で求まります。

<mark>練習5</mark> 上の解析をもとに、横軸を周波数としたプロットに改めなさい。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import wave

wf = wave.open('piano.wav', 'r')

data = wf.readframes(-1)
data = np.frombuffer(data, dtype=np.int16)

N = len(data[::2])
fft = np.fft.fft(data[::2])

k = np.arange(0, N)
nu = 44100 * k / N
plt.plot(nu, np.abs(fft))

plt.xlim(0, 1500)

🗒️ numpyのfftモジュールには、データ数とサンプリング周波数から$k$の値と周波数を対応させるリストを作る機能があるので、それを利用することもできます。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import wave

wf = wave.open('piano.wav', 'r')

# サンプリング周波数(44100Hz)を取得
sf = wf.getframerate()

data = wf.readframes(-1)
data = np.frombuffer(data, dtype=np.int16)

# データ数を取得
N = len(data[::2])
# FFTを実行
fft = np.fft.fft(data[::2])
# 振幅を求める
amp = np.abs(fft)
# 周波数を求める
freq = np.fft.fftfreq(N, 1/sf)

# 最初の方だけ表示(1500Hzぐらいまで)
plt.plot(freq[:N//2], amp[:N//2])
plt.xlim(0, 1500)
plt.xlabel('Frequency [Hz]')
plt.ylabel('Amplitude')
plt.title('Piano sound FFT')

440Hz(中音のラの音)との倍数あたりにピークが見えます。